In [1]:
# =============================================================================
# ESGF CMIP6 Full Discovery + Download
#
# Workflow:
#   1. Read available_cmip6_data.xlsx (produced by the catalog checker script)
#      to know which model/variable combos already exist on NCAR GLADE.
#   2. Query ESGF live to find ALL models that have both si AND no3
#      in Omon historical — no hardcoded model list.
#   3. Subtract what is already on GLADE.
#   4. Download missing files to scratch.
#
# Run on Casper: python esgf_download.py
# Requires:      available_cmip6_data.xlsx in the same directory
# No ESGF account required — all CMIP6 historical data is publicly accessible.
# =============================================================================

import os
import time
import requests
import pandas as pd

In [5]:
# =============================================================================
# CONFIGURATION
# =============================================================================

SAVE_DIR      = "/glade/derecho/scratch/diegovar/cmip6"
CATALOG_EXCEL = "available_cmip6_data.xlsx"   # output of catalog checker script

os.makedirs(SAVE_DIR, exist_ok=True)

# Search parameters — must match catalog checker
EXPERIMENT_ID = "historical"
TABLE_ID      = "Omon"
FREQUENCY     = "mon"
VARIABLES     = ["si", "no3"] # "phydiat", "phydiaz", "phyc", 'po4', 'tos', 'dfe','intpp', 'mlost', 'bsi', 'sst']   # only variables needed for Si*

# Member IDs to try in order — stop at first hit per model
MEMBER_ID_LIST = [
   # "r1i1p1f1",
    "r1i1p1f2",
    #"r1i1p2f1",
    #"r1i1p1f3",
]

# ESGF nodes to try in order
ESGF_NODES = [
    "https://esgf-node.llnl.gov/esg-search/search",
    "https://esgf-data.dkrz.de/esg-search/search",
    "https://esgf.nci.org.au/esg-search/search",
    "https://esgf-index1.ceda.ac.uk/esg-search/search",
]

# Download settings
MAX_RETRIES = 3
RETRY_DELAY = 5
TIMEOUT     = 60
CHUNK_SIZE  = 1024 * 1024   # 1 MB

# Only download files whose period overlaps with or comes after this year.
# CMIP6 filenames encode their period as e.g. "195001-200012" or "185001-194912".
# Files that end entirely before DATE_FILTER_START are skipped.
DATE_FILTER_START = 1950

In [6]:
# =============================================================================
# STEP 1 — READ EXCEL TO FIND WHAT IS ALREADY ON GLADE
# =============================================================================

def read_glade_availability(excel_path):
    """
    Read available_cmip6_data.xlsx and return a set of (model, variable)
    tuples that are already present on NCAR GLADE.
    Also returns the full set of models found in the catalog.
    """
    if not os.path.exists(excel_path):
        raise FileNotFoundError(
            f"Catalog Excel not found: {excel_path}\n"
            f"Please run the catalog checker script first to generate it."
        )

    df = pd.read_excel(excel_path)

    # Normalise column names — handle minor variations
    df.columns = [c.strip().lower() for c in df.columns]

    # Build set of (model, variable) already on GLADE
    on_glade = set(zip(df["model"], df["variable"]))

    # Full model list found in catalog
    models_in_catalog = set(df["model"].unique())

    print(f"  Excel loaded: {len(df)} rows, "
          f"{len(models_in_catalog)} models, "
          f"{len(on_glade)} model/variable combos on GLADE")

    return on_glade, models_in_catalog


# =============================================================================
# STEP 2 — ESGF DISCOVERY
# =============================================================================

def _query_esgf(node, params):
    """Raw ESGF REST query. Returns parsed JSON or None on failure."""
    try:
        resp = requests.get(node, params=params, timeout=TIMEOUT)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"    Node {node.split('/')[2]} failed: {e}")
        return None


def get_all_models_with_variable(variable):
    """
    Query ESGF facets to get the full list of source_ids that have
    the given variable in Omon historical.
    Returns a set of model name strings.
    """
    params = {
        "type":          "Dataset",
        "project":       "CMIP6",
        "variable_id":   variable,
        "experiment_id": EXPERIMENT_ID,
        "table_id":      TABLE_ID,
        "frequency":     FREQUENCY,
        "format":        "application/solr+json",
        "limit":         0,
        "facets":        "source_id",
        "distrib":       "true",
    }

    for node in ESGF_NODES:
        data = _query_esgf(node, params)
        if data is None:
            continue
        facets    = data.get("facet_counts", {}).get("facet_fields", {})
        raw       = facets.get("source_id", [])
        # ESGF returns [name, count, name, count, ...]
        models    = {raw[i] for i in range(0, len(raw), 2)}
        if models:
            print(f"  {len(models)} models with {variable} "
                  f"(via {node.split('/')[2]})")
            return models

    print(f"  WARNING: Could not retrieve model list for {variable} from any node")
    return set()


# =============================================================================
# STEP 3 — FILE SEARCH + DOWNLOAD
# =============================================================================

def search_files(model, variable, member_id, node):
    """Search for actual files. Returns [(filename, url, size), ...]."""
    params = {
        "type":          "File",
        "project":       "CMIP6",
        "source_id":     model,
        "variable_id":   variable,
        "experiment_id": EXPERIMENT_ID,
        "table_id":      TABLE_ID,
        "member_id":     member_id,
        "frequency":     FREQUENCY,
        "format":        "application/solr+json",
        "limit":         500,
        "fields":        "title,url,size",
        "distrib":       "true",
    }
    data = _query_esgf(node, params)
    if data is None:
        return []

    results = []
    for doc in data.get("response", {}).get("docs", []):
        title = doc.get("title", "unknown")
        size  = doc.get("size", 0)
        for url_entry in doc.get("url", []):
            parts = url_entry.split("|")
            if len(parts) == 3 and parts[2] == "HTTPServer":
                results.append((title, parts[0], size))
                break
    return results


def find_files_for_model(model, variable):
    """Try each member_id × node until files are found."""
    for member_id in MEMBER_ID_LIST:
        for node in ESGF_NODES:
            files = search_files(model, variable, member_id, node)
            if files:
                print(f"    Found {len(files)} files — "
                      f"member={member_id}, node={node.split('/')[2]}")
                return member_id, files
            time.sleep(0.3)
    return None, []


def file_is_after_cutoff(filename, cutoff_year):
    """
    Return True if the file's time period overlaps with or starts at/after
    cutoff_year.  CMIP6 filenames contain a date range like:
        si_Omon_CESM2_historical_r1i1p1f1_gn_185001-200912.nc
    We extract the END year of the range. If the file ends before cutoff_year,
    we skip it (it contains no data we need). If there is no date range in the
    filename, we keep the file to be safe.
    """
    import re
    # Match patterns like 185001-200912 or 1850-2009 (some models use 4-digit)
    match = re.search(r'_(\d{4,6})-(\d{4,6})\.nc', filename)
    if not match:
        return True   # no date info — keep to be safe
    end_str = match.group(2)
    # Parse end year — first 4 digits regardless of whether it's YYYYMM or YYYY
    end_year = int(end_str[:4])
    return end_year >= cutoff_year


def download_file(url, dest_path):
    """Stream-download with retry. Returns True on success."""
    if os.path.exists(dest_path):
        return True

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, stream=True, timeout=TIMEOUT)
            resp.raise_for_status()
            tmp = dest_path + ".tmp"
            with open(tmp, "wb") as f:
                for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        f.write(chunk)
            os.rename(tmp, dest_path)
            mb = os.path.getsize(dest_path) / 1e6
            print(f"      ✓ {os.path.basename(dest_path)}  ({mb:.1f} MB)")
            return True
        except Exception as e:
            print(f"      Attempt {attempt}/{MAX_RETRIES} failed: {e}")
            if os.path.exists(dest_path + ".tmp"):
                os.remove(dest_path + ".tmp")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)

    print(f"      ✗ FAILED: {os.path.basename(dest_path)}")
    return False


# =============================================================================
# MAIN
# =============================================================================

def run():

    # ── Step 1: read Excel to find what is already on GLADE ──────────────────
    print("=" * 60)
    print("Step 1: Reading GLADE catalog from Excel")
    print("=" * 60)
    on_glade, models_in_catalog = read_glade_availability(CATALOG_EXCEL)

    # Report per variable what is already covered
    for var in VARIABLES:
        have = {m for (m, v) in on_glade if v == var}
        print(f"  {var}: {len(have)} models already on GLADE → {sorted(have)}")

    # ── Step 2: discover all models on ESGF with si AND no3 ──────────────────
    print("\n" + "=" * 60)
    print("Step 2: Querying ESGF for all models with si AND no3")
    print("=" * 60)

    models_with_si  = get_all_models_with_variable("si")
    models_with_no3 = get_all_models_with_variable("no3")
    models_with_both = models_with_si & models_with_no3

    print(f"\n  Models on ESGF with both si and no3: {len(models_with_both)}")
    for m in sorted(models_with_both):
        print(f"    {m}")

    # ── Step 3: build the download plan ──────────────────────────────────────
    # For each (model, variable) combo: download if on ESGF but NOT on GLADE
    download_plan = []
    for model in sorted(models_with_both):
        for var in VARIABLES:
            if (model, var) not in on_glade:
                download_plan.append((model, var))

    print(f"\n  Download plan: {len(download_plan)} model/variable combos")
    for model, var in download_plan:
        print(f"    {model:30s}  {var}")

    if not download_plan:
        print("\nNothing to download — all models already on GLADE.")
        return

    # ── Step 4: download ─────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("Step 3: Downloading")
    print("=" * 60)

    failed  = []
    summary = {}

    for model, var in download_plan:
        key = (model, var)
        print(f"\n  {model} / {var}")

        out_dir = os.path.join(SAVE_DIR, model, var)
        os.makedirs(out_dir, exist_ok=True)

        _, files = find_files_for_model(model, var)

        if not files:
            print(f"    No files found on ESGF for {model}/{var}")
            summary[key] = 0
            continue

        n_ok = 0
        n_skipped = 0
        for fname, url, size in files:
            # Skip files that end entirely before DATE_FILTER_START
            if not file_is_after_cutoff(fname, DATE_FILTER_START):
                n_skipped += 1
                continue
            dest = os.path.join(out_dir, fname)
            ok   = download_file(url, dest)
            if ok:
                n_ok += 1
            else:
                failed.append((model, var, fname, url))

        summary[key] = n_ok
        print(f"    {n_ok}/{len(files)} files downloaded "
              f"({n_skipped} skipped — before {DATE_FILTER_START})")

    # ── Final report ─────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    for (model, var), n in sorted(summary.items()):
        tag = "✓" if n > 0 else "✗ NOT FOUND"
        print(f"  {tag}  {model:30s}  {var:5s}  {n} files")

    if failed:
        retry_path = os.path.join(SAVE_DIR, "retry_failed.sh")
        with open(retry_path, "w") as f:
            f.write("#!/bin/bash\n# Retry failed downloads\n\n")
            for model, var, fname, url in failed:
                out_dir = os.path.join(SAVE_DIR, model, var)
                f.write(f"wget -P '{out_dir}' -c '{url}'\n")
        os.chmod(retry_path, 0o755)
        print(f"\n  {len(failed)} files failed → retry script: {retry_path}")
        print("  Run with:  bash retry_failed.sh")

    print("\nDone.")

In [7]:
# =============================================================================
# RUN
# =============================================================================

run()

Step 1: Reading GLADE catalog from Excel
  Excel loaded: 196 rows, 22 models, 196 model/variable combos on GLADE
  si: 11 models already on GLADE → ['CESM2', 'CESM2-FV2', 'CESM2-WACCM', 'CESM2-WACCM-FV2', 'CNRM-ESM2-1', 'GFDL-ESM4', 'GISS-E2-1-G-CC', 'IPSL-CM6A-LR', 'NorCPM1', 'NorESM2-LM', 'UKESM1-0-LL']
  no3: 12 models already on GLADE → ['CESM2', 'CESM2-FV2', 'CESM2-WACCM', 'CESM2-WACCM-FV2', 'CNRM-ESM2-1', 'CanESM5', 'GFDL-ESM4', 'GISS-E2-1-G-CC', 'IPSL-CM6A-LR', 'NorCPM1', 'NorESM2-LM', 'UKESM1-0-LL']

Step 2: Querying ESGF for all models with si AND no3
  19 models with si (via esgf-node.llnl.gov)
  27 models with no3 (via esgf-node.llnl.gov)

  Models on ESGF with both si and no3: 19
    CESM2
    CESM2-FV2
    CESM2-WACCM
    CESM2-WACCM-FV2
    CMCC-ESM2
    CNRM-ESM2-1
    GFDL-ESM4
    GISS-E2-1-G
    GISS-E2-1-G-CC
    IPSL-CM5A2-INCA
    IPSL-CM6A-LR
    IPSL-CM6A-LR-INCA
    MPI-ESM-1-2-HAM
    MPI-ESM1-2-HR
    MPI-ESM1-2-LR
    NorCPM1
    NorESM2-LM
    NorESM2-MM
   